In [ ]:
# Prediction of COVID-19 Disease Severity
## Research Paper Replication

### Dataset Source
Mexican COVID-19 patient pre-condition dataset

### Objective
To predict COVID-19 severity using chronic disease indicators

In [3]:
#Importing Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
#Loading Dataset
data = pd.read_csv('Dataset\covid.csv')
data.shape

data.info()
data.head()
data.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566602 entries, 0 to 566601
Data columns (total 23 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   id                   566602 non-null  object
 1   sex                  566602 non-null  int64 
 2   patient_type         566602 non-null  int64 
 3   entry_date           566602 non-null  object
 4   date_symptoms        566602 non-null  object
 5   date_died            566602 non-null  object
 6   intubed              566602 non-null  int64 
 7   pneumonia            566602 non-null  int64 
 8   age                  566602 non-null  int64 
 9   pregnancy            566602 non-null  int64 
 10  diabetes             566602 non-null  int64 
 11  copd                 566602 non-null  int64 
 12  asthma               566602 non-null  int64 
 13  inmsupr              566602 non-null  int64 
 14  hypertension         566602 non-null  int64 
 15  other_disease        566602 non-nu

Index(['id', 'sex', 'patient_type', 'entry_date', 'date_symptoms', 'date_died',
       'intubed', 'pneumonia', 'age', 'pregnancy', 'diabetes', 'copd',
       'asthma', 'inmsupr', 'hypertension', 'other_disease', 'cardiovascular',
       'obesity', 'renal_chronic', 'tobacco', 'contact_other_covid',
       'covid_res', 'icu'],
      dtype='object')

In [ ]:
#Data Cleaning and Preprocessing
#Feature Selection
selected_columns = [
    'age',
    'sex',
    'date_died',
    'diabetes',
    'obesity',
    'covid_res'
]

data_selected = data[selected_columns]
data_selected.head()

data_selected = data[selected_columns].copy()

#Data Cleaning
invalid_values = [97, 98, 99]

for col in ['diabetes', 'obesity', 'covid_res']:
    data_selected = data_selected[~data_selected[col].isin(invalid_values)]

data_selected.shape

data_selected['death'] = data_selected['date_died'].apply(
    lambda x: 0 if x == '9999-99-99' else 1
)

data_selected = data_selected.drop(columns=['date_died'])

data_selected.head()

#Selecting sample 10000 patients data
data_sample = data_selected.sample(n=10000, random_state=42)
data_sample.shape


(10000, 6)

In [17]:
#Feature Encoding and Train-Test Split
#Define Features and Target Variable
X = data_sample.drop(columns=['death'])
y = data_sample['death']

X.shape, y.shape

#Train-Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape


((8000, 5), (2000, 5))

In [20]:
#Model Training and Evaluation
#Importing Required Libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

#Function to Train and Evaluate Models
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred)
    }

#Train all Models
#1. Random Forest
rf = RandomForestClassifier(random_state=42)
rf_results = evaluate_model(rf, X_train, X_test, y_train, y_test)
rf_results

#2. Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt_results = evaluate_model(dt, X_train, X_test, y_train, y_test)
dt_results

#3. Support Vector Machine
svm = SVC()
svm_results = evaluate_model(svm, X_train, X_test, y_train, y_test)
svm_results

#4. K-Nearest Neighbors
knn = KNeighborsClassifier()
knn_results = evaluate_model(knn, X_train, X_test, y_train, y_test)
knn_results

#5. Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr_results = evaluate_model(lr, X_train, X_test, y_train, y_test)
lr_results

#Combine Results
results_df = pd.DataFrame({
    "Random Forest": rf_results,
    "Decision Tree": dt_results,
    "SVM": svm_results,
    "KNN": knn_results,
    "Logistic Regression": lr_results
}).T

results_df
#Combining Results
results_df = pd.DataFrame({
    "Random Forest": rf_results,
    "Decision Tree": dt_results,
    "SVM": svm_results,
    "KNN": knn_results,
    "Logistic Regression": lr_results
}).T

results_df




c:\Users\Raag\OneDrive\Desktop\Personal\ML Mini Project\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,Accuracy,Precision,Recall,F1-score
Random Forest,0.9305,0.175000,0.061947,0.091503
Decision Tree,0.9300,0.135135,0.044248,0.066667
SVM,0.9435,0.000000,0.000000,0.000000
KNN,0.9350,0.257143,0.079646,0.121622
Logistic Regression,0.9400,0.315789,0.053097,0.090909
